
<span style="font-size:12px;">
Do NOT run this whole file as one cell or with exec() or !python.

The cell boundaries are marked with: # — CELL N —
</span>

In [ ]:
# ── CELL 1 ── Mount Drive (run this cell alone first, complete the auth popup)
from google.colab import drive
drive.mount('/content/drive')

# After mounting, verify it worked before continuing:
import os
assert os.path.exists('/content/drive/MyDrive'), \
    "Drive not mounted! Re-run this cell and complete the auth."
print("Drive mounted OK. MyDrive contents:", os.listdir('/content/drive/MyDrive')[:5])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted OK. MyDrive contents: ['Essay 4_mepi copy.docx', 'Colab Notebooks', 'Classroom', 'Google AI Studio', 'Hamra Night Commute Safety Survey.gform']


In [ ]:
# ── CELL 2 ── Install dependencies
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "datasets", "pillow", "imagehash", "tqdm", "pandas",
     "requests", "torchvision"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip install FAILED:", result.stderr)
else:
    print("Dependencies installed OK")

Dependencies installed OK


In [ ]:
# ── CELL 3 ── Configuration
CONFIG = {
    "mode": "medium",
    "budgets": {
        "small":  500,
        "medium": 6000,
        "large":  10000,
    },
    "source_fractions": {
        "imagenet":        0.17,
        "places365":       0.18,
        "coco":            0.30,
        "british_library": 0.15,
        "loc":             0.15,
        "flickr_commons":  0.05,
    },
    "split": {"train": 0.80, "val": 0.10, "test": 0.10},
    "min_width":  128,
    "min_height": 128,
    "max_aspect": 4.0,
    "exact_dedup":      True,
    "perceptual_dedup": False,
    "phash_threshold":  8,
    "edge_case_boost":   True,
    "res_tier_high_min": 512,
    "imagenet_hf_dataset":  "yujiepan/imagenet-1k-train-sampled2048",
    "places365_hf_dataset": "dpdl-benchmark/places365-mini-sample-hard",
    "coco_hf_dataset":      "nickpai/coco2017-colorization",
    "flickr_commons_root":  "/content/drive/MyDrive/datasets/flickr_commons/",
    "loc_root":             "/content/drive/MyDrive/datasets/loc/",
    "loc_api_key":          "",
    "british_library_root": "/content/drive/MyDrive/datasets/british_library/",
    "british_library_hf":   "huggan/wikiart",
    "output_root":   "/content/drive/MyDrive/merged_dataset/",
    "images_subdir": "images",
    "manifest_name": "merged_manifest",
    "summary_name":  "dataset_summary",
    "log_file":      "pipeline.log",
}

frac_sum = sum(CONFIG["source_fractions"].values())
assert abs(frac_sum - 1.0) < 0.001, f"source_fractions must sum to 1.0, got {frac_sum}"
print(f"Config OK — mode={CONFIG['mode']}, budget={CONFIG['budgets'][CONFIG['mode']]}")

Config OK — mode=medium, budget=6000


In [ ]:
# ── CELL 4 ── Imports, logging, folder structure
import os, sys, json, csv, hashlib, logging, random, shutil, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from PIL import Image
import imagehash
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def setup_logging(cfg: dict) -> logging.Logger:
    log_path = Path(cfg["output_root"]) / cfg["log_file"]
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("pipeline")
    logger.setLevel(logging.DEBUG)
    # Remove old handlers to avoid duplicate output on re-run
    logger.handlers.clear()
    fh = logging.FileHandler(str(log_path))
    fh.setLevel(logging.DEBUG)
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    fmt = logging.Formatter("[%(asctime)s] %(levelname)s — %(message)s", "%H:%M:%S")
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


def create_folder_structure(cfg: dict, log) -> Dict[str, Path]:
    root = Path(cfg["output_root"])
    paths: Dict[str, Path] = {
        "root":      root,
        "images":    root / cfg["images_subdir"],
        "manifests": root / "manifests",
        "splits":    root / "splits",
        "logs":      root / "logs",
    }
    for source in cfg["source_fractions"]:
        paths[f"img_{source}"] = paths["images"] / source
    for name, p in paths.items():
        p.mkdir(parents=True, exist_ok=True)
        # Verify the folder actually exists on Drive
        if not p.exists():
            raise RuntimeError(f"Failed to create folder {p} — Drive may not be writable")
    log.info(f"All folders created under: {root}")
    # Write a test file to confirm Drive writes work
    test_file = root / "_write_test.txt"
    test_file.write_text("write test OK")
    assert test_file.exists(), "Write test failed — Drive is not writable!"
    test_file.unlink()
    log.info("Drive write test PASSED")
    return paths


# Initialise now (Drive is already mounted from Cell 1)
log  = setup_logging(CONFIG)
PATHS = create_folder_structure(CONFIG, log)
print("\nFolder structure:")
for name, p in PATHS.items():
    print(f"  {name}: {p}")


[13:02:53] INFO — All folders created under: /content/drive/MyDrive/merged_dataset


INFO:pipeline:All folders created under: /content/drive/MyDrive/merged_dataset


[13:02:53] INFO — Drive write test PASSED


INFO:pipeline:Drive write test PASSED



Folder structure:
  root: /content/drive/MyDrive/merged_dataset
  images: /content/drive/MyDrive/merged_dataset/images
  manifests: /content/drive/MyDrive/merged_dataset/manifests
  splits: /content/drive/MyDrive/merged_dataset/splits
  logs: /content/drive/MyDrive/merged_dataset/logs
  img_imagenet: /content/drive/MyDrive/merged_dataset/images/imagenet
  img_places365: /content/drive/MyDrive/merged_dataset/images/places365
  img_coco: /content/drive/MyDrive/merged_dataset/images/coco
  img_british_library: /content/drive/MyDrive/merged_dataset/images/british_library
  img_loc: /content/drive/MyDrive/merged_dataset/images/loc
  img_flickr_commons: /content/drive/MyDrive/merged_dataset/images/flickr_commons


In [ ]:
# ── CELL 5 ── Utility functions
def compute_md5(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def compute_phash(path: str) -> Optional[imagehash.ImageHash]:
    try:
        return imagehash.phash(Image.open(path).convert("RGB"))
    except Exception:
        return None


def validate_image(path: str, cfg: dict) -> Tuple[bool, dict]:
    meta: dict = {
        "width": None, "height": None, "aspect_ratio": None,
        "color_mode": None, "file_size": None, "extreme_aspect": False,
    }
    try:
        meta["file_size"] = os.path.getsize(path)
        with Image.open(path) as img:
            w, h = img.size
            meta["width"]          = w
            meta["height"]         = h
            meta["color_mode"]     = img.mode
            meta["aspect_ratio"]   = round(w / h, 3)
            meta["extreme_aspect"] = (max(w, h) / min(w, h)) > cfg["max_aspect"]
            if w < cfg["min_width"] or h < cfg["min_height"]:
                return False, meta
        return True, meta
    except Exception as e:
        log.debug(f"Unreadable image {path}: {e}")
        return False, meta


def safe_copy(src: str, dst: str) -> bool:
    try:
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True
    except Exception as e:
        log.debug(f"Copy failed {src} → {dst}: {e}")
        return False


def sample_paths(paths: List[str], quota: int, seed: int = 42) -> List[str]:
    random.seed(seed)
    return random.sample(paths, min(quota, len(paths)))


def hf_to_pil(val) -> Optional[Image.Image]:
    if val is None:
        return None
    if isinstance(val, Image.Image):
        return val
    try:
        import numpy as np
        return Image.fromarray(val)
    except Exception:
        return None


def probe_hf_column(row: dict, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in row and row[c] is not None:
            return c
    return None


print("Utility functions defined OK")



Utility functions defined OK


In [ ]:
# ── CELL 6 ── Dataset loaders

def load_imagenet(cfg: dict, quota: int) -> List[dict]:
    try:
        from datasets import load_dataset
    except ImportError:
        log.warning("datasets not installed — skipping ImageNet")
        return []
    dataset_name = cfg.get("imagenet_hf_dataset", "yujiepan/imagenet-1k-train-sampled2048")
    log.info(f"ImageNet: loading {dataset_name}")
    try:
        ds = load_dataset(dataset_name, split="train")
    except Exception as e:
        log.warning(f"ImageNet HF load failed: {e}")
        return []
    if len(ds) == 0:
        log.warning("ImageNet HF dataset is empty")
        return []
    sample_row = ds[0]
    log.info(f"ImageNet columns: {list(sample_row.keys())}")
    img_col   = probe_hf_column(sample_row, ["image", "img", "pixel_values"])
    id_col    = probe_hf_column(sample_row, ["image_id", "id", "idx", "path"])
    label_col = probe_hf_column(sample_row, ["label", "label_name", "class", "synset"])
    path_col  = probe_hf_column(sample_row, ["path", "file_name", "filename"])
    if img_col is None:
        log.warning("ImageNet HF: no image column found — skipping")
        return []
    indices = list(range(len(ds)))
    random.seed(42); random.shuffle(indices)
    img_dir = PATHS["img_imagenet"]
    records: List[dict] = []
    saved = 0
    for i in tqdm(indices[:quota], desc="ImageNet"):
        row = ds[i]
        img = hf_to_pil(row.get(img_col))
        if img is None:
            continue
        raw_id    = row.get(id_col, i) if id_col else i
        raw_label = row.get(label_col) if label_col else None
        raw_path  = row.get(path_col) if path_col else None
        safe_id   = str(raw_id).replace("/", "_").replace("\\", "_").replace(" ", "_")
        dst = str(img_dir / f"imagenet_{safe_id}.jpg")
        try:
            img.convert("RGB").save(dst, quality=95)
            saved += 1
        except Exception as e:
            log.warning(f"ImageNet save failed idx {i}: {e}")
            continue
        subset_name = raw_path.split("/")[0] if raw_path and isinstance(raw_path, str) else "hf_subset"
        records.append({
            "original_path_or_url":  f"hf://{dataset_name}#{raw_id}",
            "source_dataset":        "imagenet",
            "source_subset":         subset_name,
            "label":                 str(raw_label) if raw_label is not None else None,
            "scene_type":            "object",
            "historical_flag":       False,
            "historical_color_flag": False,
            "damaged_flag":          False,
            "license_info":          "ImageNet via HuggingFace — research use",
            "institution":           "ImageNet / HuggingFace",
            "decade":                None,
            "notes":                 f"HF source: {dataset_name}",
            "_saved_path":           dst,
        })
    log.info(f"ImageNet: {saved} files saved to {img_dir}")
    return records


def load_places365(cfg: dict, quota: int) -> List[dict]:
    try:
        from datasets import load_dataset
    except ImportError:
        log.warning("datasets not installed — skipping Places365")
        return []
    dataset_name = cfg.get("places365_hf_dataset", "dpdl-benchmark/places365-mini-sample-hard")
    log.info(f"Places365: loading {dataset_name}")
    try:
        ds = load_dataset(dataset_name, split="train")
    except Exception as e:
        log.warning(f"Places365 HF load failed: {e}")
        return []
    if len(ds) == 0:
        log.warning("Places365 HF dataset is empty")
        return []
    sample_row = ds[0]
    log.info(f"Places365 columns: {list(sample_row.keys())}")
    img_col   = probe_hf_column(sample_row, ["image", "img", "pixel_values"])
    label_col = probe_hf_column(sample_row, ["label", "scene", "class", "category"])
    path_col  = probe_hf_column(sample_row, ["path", "file_name", "filename"])
    if img_col is None:
        log.warning("Places365 HF: no image column found — skipping")
        return []
    indices = list(range(len(ds)))
    random.seed(42); random.shuffle(indices)
    img_dir = PATHS["img_places365"]
    records: List[dict] = []
    saved = 0
    for i in tqdm(indices[:quota], desc="Places365"):
        row = ds[i]
        img = hf_to_pil(row.get(img_col))
        if img is None:
            continue
        raw_label = row.get(label_col) if label_col else None
        raw_path  = row.get(path_col) if path_col else None
        dst = str(img_dir / f"places365_{i}.jpg")
        try:
            img.convert("RGB").save(dst, quality=95)
            saved += 1
        except Exception as e:
            log.warning(f"Places365 save failed idx {i}: {e}")
            continue
        subset_name = raw_path.split("/")[0] if raw_path and isinstance(raw_path, str) else "hf_subset"
        records.append({
            "original_path_or_url":  f"hf://{dataset_name}#{i}",
            "source_dataset":        "places365",
            "source_subset":         subset_name,
            "label":                 str(raw_label) if raw_label is not None else None,
            "scene_type":            "scene",
            "historical_flag":       False,
            "historical_color_flag": False,
            "damaged_flag":          False,
            "license_info":          "Places365 via HuggingFace",
            "institution":           "MIT CSAIL / HuggingFace",
            "decade":                None,
            "notes":                 f"HF source: {dataset_name}",
            "_saved_path":           dst,
        })
    log.info(f"Places365: {saved} files saved to {img_dir}")
    return records


def load_coco(cfg: dict, quota: int) -> List[dict]:
    try:
        from datasets import load_dataset
    except ImportError:
        log.warning("datasets not installed — skipping COCO")
        return []
    from io import BytesIO
    import requests as req
    log.info(f"COCO: loading {cfg['coco_hf_dataset']}")
    try:
        ds = load_dataset(cfg["coco_hf_dataset"], split="train")
    except Exception as e:
        log.warning(f"COCO load failed: {e}")
        return []
    if len(ds) == 0:
        log.warning("COCO dataset is empty")
        return []
    sample_row = ds[0]
    log.info(f"COCO columns: {list(sample_row.keys())}")
    img_col   = probe_hf_column(sample_row, ["image", "img", "pixel_values"])
    id_col    = probe_hf_column(sample_row, ["image_id", "id", "idx"])
    label_col = probe_hf_column(sample_row, ["category", "label", "categories", "supercategory"])
    url_col   = probe_hf_column(sample_row, ["coco_url", "flickr_url", "url"])
    if img_col is None and url_col is None:
        log.warning("COCO: no image or URL column — skipping")
        return []
    indices = list(range(len(ds)))
    random.seed(42); random.shuffle(indices)
    img_dir = PATHS["img_coco"]
    records: List[dict] = []
    saved = 0
    for i in tqdm(indices[:quota], desc="COCO"):
        row = ds[i]
        img = None
        if img_col is not None:
            img = hf_to_pil(row.get(img_col))
        elif url_col is not None:
            try:
                r = req.get(row[url_col], timeout=20)
                r.raise_for_status()
                img = Image.open(BytesIO(r.content)).convert("RGB")
            except Exception as e:
                log.debug(f"COCO URL download failed idx {i}: {e}")
                continue
        if img is None:
            continue
        img_id = row.get(id_col, i) if id_col else i
        dst = str(img_dir / f"coco_{img_id}.jpg")
        try:
            img.convert("RGB").save(dst)
            saved += 1
        except Exception as e:
            log.warning(f"COCO save failed idx {i}: {e}")
            continue
        records.append({
            "original_path_or_url":  row.get(url_col, f"hf://{cfg['coco_hf_dataset']}#{img_id}") if url_col else f"hf://{cfg['coco_hf_dataset']}#{img_id}",
            "source_dataset":        "coco",
            "source_subset":         "colorization",
            "label":                 str(row.get(label_col, "")) if label_col else "",
            "scene_type":            "mixed",
            "historical_flag":       False,
            "historical_color_flag": False,
            "damaged_flag":          False,
            "license_info":          "COCO — CC BY 4.0",
            "institution":           "Microsoft",
            "decade":                None,
            "notes":                 "",
            "_saved_path":           dst,
        })
    log.info(f"COCO: {saved} files saved to {img_dir}")
    return records


def load_flickr_commons(cfg: dict, quota: int) -> List[dict]:
    root = Path(cfg["flickr_commons_root"])
    if not root.exists():
        log.warning(f"Flickr Commons not found: {root} — skipping")
        return []
    all_imgs: List[str] = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        all_imgs.extend(str(p) for p in root.rglob(ext))
    if not all_imgs:
        log.warning("Flickr Commons folder is empty — skipping")
        return []
    chosen = sample_paths(all_imgs, quota)
    records = []
    for p in chosen:
        institution = Path(p).parent.name
        records.append({
            "original_path_or_url":  p,
            "source_dataset":        "flickr_commons",
            "source_subset":         institution,
            "label":                 None,
            "scene_type":            "historical",
            "historical_flag":       True,
            "historical_color_flag": None,
            "damaged_flag":          None,
            "license_info":          "No Known Copyright Restrictions",
            "institution":           institution,
            "decade":                None,
            "notes":                 "Verify individual licenses at Flickr Commons",
        })
    log.info(f"Flickr Commons: {len(records)} selected")
    return records


def load_loc(cfg: dict, quota: int) -> List[dict]:
    root = Path(cfg["loc_root"])
    root.mkdir(parents=True, exist_ok=True)
    local_imgs: List[str] = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        local_imgs.extend(str(p) for p in root.rglob(ext))
    if len(local_imgs) >= quota:
        log.info(f"LOC: using {quota} local images")
        return _build_loc_records(sample_paths(local_imgs, quota))
    log.info("LOC: fetching via public API…")
    return _loc_api_download(cfg, quota)


def _loc_api_download(cfg: dict, quota: int) -> List[dict]:
    """
    Download from the LOC /photos/ endpoint with pagination and User-Agent header.
    """
    import requests as req
    root = Path(cfg["loc_root"])

    queries = [
        "photograph portrait",
        "civil war",
        "great depression photograph",
        "street scene 1900",
        "landscape archive photograph",
    ]

    records:    List[dict] = []
    downloaded: int        = 0

    for query in queries:
        if downloaded >= quota:
            break
        for page in range(1, 4):
            if downloaded >= quota:
                break
            try:
                resp = req.get(
                    "https://www.loc.gov/photos/",
                    params={"q": query, "fo": "json", "c": 25, "sp": page},
                    timeout=30,
                    headers={"User-Agent": "Mozilla/5.0 (research pipeline)"},
                )
                if resp.status_code != 200:
                    log.debug(f"LOC HTTP {resp.status_code} for '{query}' page {page}")
                    break
                data  = resp.json()
                items = data.get("results", [])
                if not items:
                    break
                log.debug(f"LOC '{query}' page {page}: {len(items)} items")
            except Exception as e:
                log.warning(f"LOC API failed '{query}' page {page}: {e}")
                break

            for item in items:
                if downloaded >= quota:
                    break
                raw_urls = item.get("image_url", [])
                if isinstance(raw_urls, str):
                    raw_urls = [raw_urls]
                thumb = item.get("thumb_gallery") or item.get("thumbnail")
                if thumb and isinstance(thumb, str):
                    raw_urls = raw_urls + [thumb]
                if not raw_urls:
                    continue

                saved_dst: Optional[str] = None
                used_url:  Optional[str] = None

                for img_url in raw_urls:
                    if not isinstance(img_url, str) or not img_url.startswith("http"):
                        continue
                    fname = f"loc_{hashlib.md5(img_url.encode()).hexdigest()[:12]}.jpg"
                    dst   = str(root / fname)
                    if Path(dst).exists():
                        saved_dst = dst; used_url = img_url; break
                    try:
                        r = req.get(img_url, timeout=20, stream=True,
                                    headers={"User-Agent": "Mozilla/5.0 (research pipeline)"})
                        if r.status_code != 200:
                            continue
                        ct = r.headers.get("Content-Type", "")
                        if "image" not in ct and "jpeg" not in ct:
                            continue
                        with open(dst, "wb") as fh:
                            for chunk in r.iter_content(4096):
                                fh.write(chunk)
                        try:
                            with Image.open(dst) as test_img:
                                test_img.verify()
                        except Exception:
                            try: os.remove(dst)
                            except Exception: pass
                            continue
                        saved_dst = dst; used_url = img_url; break
                    except Exception as e:
                        log.debug(f"LOC download failed {img_url}: {e}")
                        continue

                if saved_dst is None:
                    continue

                decade_str = str(item.get("date", ""))[:4]
                try:
                    decade: Optional[str] = f"{(int(decade_str) // 10) * 10}s"
                except Exception:
                    decade = None

                records.append({
                    "original_path_or_url":  used_url,
                    "source_dataset":        "loc",
                    "source_subset":         query,
                    "label":                 None,
                    "scene_type":            "historical",
                    "historical_flag":       True,
                    "historical_color_flag": None,
                    "damaged_flag":          None,
                    "license_info":          "Public Domain / No Known Copyright",
                    "institution":           "Library of Congress",
                    "decade":                decade,
                    "notes":                 item.get("title", ""),
                    "_saved_path":           saved_dst,
                })
                downloaded += 1

    log.info(f"LOC: {len(records)} fetched (quota {quota})")
    return records
def _build_loc_records(paths: List[str]) -> List[dict]:
    return [{
        "original_path_or_url":  p,
        "source_dataset":        "loc",
        "source_subset":         "local",
        "label":                 None,
        "scene_type":            "historical",
        "historical_flag":       True,
        "historical_color_flag": None,
        "damaged_flag":          None,
        "license_info":          "Public Domain / No Known Copyright",
        "institution":           "Library of Congress",
        "decade":                None,
        "notes":                 "",
    } for p in paths]


def load_british_library(cfg: dict, quota: int) -> List[dict]:
    root = Path(cfg["british_library_root"])
    root.mkdir(parents=True, exist_ok=True)
    local_imgs: List[str] = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        local_imgs.extend(str(p) for p in root.rglob(ext))
    if len(local_imgs) >= quota:
        chosen = sample_paths(local_imgs, quota)
        log.info(f"British Library: {len(chosen)} local images used")
        return [{
            "original_path_or_url":  p,
            "source_dataset":        "british_library",
            "source_subset":         "local",
            "label":                 None,
            "scene_type":            "historical",
            "historical_flag":       True,
            "historical_color_flag": None,
            "damaged_flag":          None,
            "license_info":          "Public Domain / CC0",
            "institution":           "British Library",
            "decade":                None,
            "notes":                 "",
        } for p in chosen]
    log.info("British Library: loading from HuggingFace…")
    try:
        from datasets import load_dataset
        ds     = load_dataset(cfg["british_library_hf"], split="train")
        sample = ds[0]
        log.info(f"British Library columns: {list(sample.keys())}")
        img_col  = probe_hf_column(sample, ["image", "img", "pixel_values"])
        year_col = probe_hf_column(sample, ["year", "date", "decade", "publication_year"])
        if img_col is None:
            log.warning("British Library HF: no image column — skipping")
            return []
        img_dir = PATHS["img_british_library"]
        indices = list(range(len(ds)))
        random.seed(42); random.shuffle(indices)
        records: List[dict] = []
        saved = 0
        for i in tqdm(indices[:quota], desc="British Library"):
            row = ds[i]
            img = hf_to_pil(row.get(img_col))
            if img is None:
                continue
            dst = str(img_dir / f"bl_{i}.jpg")
            try:
                img.convert("RGB").save(dst)
                saved += 1
            except Exception as e:
                log.warning(f"British Library save failed idx {i}: {e}")
                continue
            decade = str(row.get(year_col, "")) if year_col else None
            records.append({
                "original_path_or_url":  f"hf://{cfg['british_library_hf']}#{i}",
                "source_dataset":        "british_library",
                "source_subset": "wikiart",
                "label":                 None,
                "scene_type":            "historical",
                "historical_flag":       True,
                "historical_color_flag": None,
                "damaged_flag":          None,
                "license_info":          "Public Domain / CC0",
                "institution":           "British Library",
                "decade":                decade or None,
                "notes":                 "",
                "_saved_path":           dst,
            })
        log.info(f"British Library: {saved} files saved to {img_dir}")
        return records
    except Exception as e:
        log.warning(f"British Library HF load failed: {e}")
        return []



print("Loaders defined OK")


Loaders defined OK


In [ ]:
# ── CELL 7 ── Processing, splitting, manifests

def resolve_local_path(record: dict) -> Optional[str]:
    if "_saved_path" in record:
        p = record["_saved_path"]
        return p if os.path.exists(p) else None
    src = record.get("original_path_or_url", "")
    if not src or src.startswith("hf://"):
        return None
    return src if os.path.exists(src) else None


def process_records(records: List[dict], cfg: dict, paths: dict) -> List[dict]:
    seen_md5:   Dict[str, str]            = {}
    seen_phash: List[imagehash.ImageHash] = []
    processed:  List[dict]               = []
    for rec in tqdm(records, desc="Validating"):
        local_path = resolve_local_path(rec)
        if local_path is None:
            log.debug(f"Unresolvable: {rec.get('original_path_or_url','?')}")
            continue
        valid, meta = validate_image(local_path, cfg)
        if not valid:
            log.debug(f"Invalid: {local_path}")
            continue
        if cfg["exact_dedup"]:
            md5 = compute_md5(local_path)
            if md5 in seen_md5:
                continue
            seen_md5[md5] = local_path
        if cfg["perceptual_dedup"]:
            ph = compute_phash(local_path)
            if ph is not None:
                if any((ph - ex) <= cfg["phash_threshold"] for ex in seen_phash):
                    continue
                seen_phash.append(ph)
        color_mode = meta["color_mode"]
        if rec.get("historical_color_flag") is None:
            rec["historical_color_flag"] = color_mode not in ("L", "1", "P")
        src_name = rec["source_dataset"]
        dest_dir = paths.get(f"img_{src_name}", paths["images"] / src_name)
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest_path = str(dest_dir / Path(local_path).name)
        if not os.path.exists(dest_path) and not safe_copy(local_path, dest_path):
            continue
        rec.update(
            image_id       = f"{src_name}_{hashlib.md5(dest_path.encode()).hexdigest()[:10]}",
            local_path     = dest_path,
            width          = meta["width"],
            height         = meta["height"],
            aspect_ratio   = meta["aspect_ratio"],
            color_mode     = color_mode,
            file_size      = meta["file_size"],
            extreme_aspect = meta["extreme_aspect"],
        )
        rec.pop("_saved_path", None)
        processed.append(rec)
    log.info(f"Validation: {len(processed)}/{len(records)} kept")
    return processed


def boost_edge_cases(records: List[dict], cfg: dict) -> List[dict]:
    if not cfg.get("edge_case_boost", True):
        return records
    high_min = cfg.get("res_tier_high_min", 512)
    hist    = [r for r in records if r.get("historical_flag")]
    general = [r for r in records if not r.get("historical_flag")]
    bw_records = [r for r in hist if r.get("color_mode") in ("L", "1", "P")]
    high_res   = [r for r in hist if min(r.get("width") or 0, r.get("height") or 0) >= high_min]
    decade_groups: Dict[str, List[dict]] = defaultdict(list)
    no_decade: List[dict] = []
    for r in hist:
        d = r.get("decade")
        (decade_groups[d] if d else no_decade).append(r)
    per_decade_cap = max(10, len(hist) // max(len(decade_groups), 1))
    decade_capped: List[dict] = []
    for d in sorted(decade_groups):
        bucket = decade_groups[d]; random.seed(42); random.shuffle(bucket)
        decade_capped.extend(bucket[:per_decade_cap])
    decade_capped.extend(no_decade)
    boosted: List[dict] = []
    seen_ids: set = set()
    for bucket in (bw_records, high_res, decade_capped, hist):
        for r in bucket:
            rid = id(r)
            if rid not in seen_ids:
                boosted.append(r); seen_ids.add(rid)
    log.info(f"Edge boost: {len(bw_records)} B&W, {len(high_res)} high-res, {len(decade_groups)} decades")
    return general + boosted


def assign_splits(records: List[dict], cfg: dict) -> List[dict]:
    train_f = cfg["split"]["train"]
    val_f   = cfg["split"]["val"]
    strata: Dict[str, List[int]] = defaultdict(list)
    for i, r in enumerate(records):
        bw_flag   = "bw" if r.get("color_mode") in ("L", "1", "P") else "color"
        hist_flag = "hist" if r.get("historical_flag") else "general"
        strata[f"{r['source_dataset']}|{bw_flag}|{hist_flag}"].append(i)
    split_map: Dict[int, str] = {}
    for stratum, idxs in strata.items():
        random.seed(42); random.shuffle(idxs)
        n = len(idxs); n_train = int(n * train_f); n_val = int(n * val_f)
        for i in idxs[:n_train]:             split_map[i] = "train"
        for i in idxs[n_train:n_train+n_val]: split_map[i] = "val"
        for i in idxs[n_train+n_val:]:       split_map[i] = "test"
    for i, r in enumerate(records):
        r["split"] = split_map.get(i, "train")
    counts = Counter(r["split"] for r in records)
    log.info(f"Splits — train:{counts['train']} val:{counts['val']} test:{counts['test']}")
    for split_name in ("train", "val", "test"):
        paths_list = [r["local_path"] for r in records if r.get("split") == split_name and r.get("local_path")]
        out = PATHS["splits"] / f"{split_name}.txt"
        out.write_text("\n".join(paths_list))
        log.info(f"  {split_name}.txt — {len(paths_list)} images")
    return records


MANIFEST_FIELDS = [
    "image_id", "source_dataset", "source_subset", "original_path_or_url",
    "local_path", "split", "width", "height", "aspect_ratio", "color_mode",
    "file_size", "label", "scene_type", "historical_flag", "historical_color_flag",
    "damaged_flag", "extreme_aspect", "license_info", "institution", "decade", "notes",
]


def save_manifests(records: List[dict], cfg: dict, paths: dict) -> None:
    mdir       = paths["manifests"]
    normalized = [{f: r.get(f) for f in MANIFEST_FIELDS} for r in records]
    csv_path   = mdir / f"{cfg['manifest_name']}.csv"
    with open(str(csv_path), "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows(normalized)
    log.info(f"CSV saved: {csv_path}")
    json_path = mdir / f"{cfg['manifest_name']}.json"
    json_path.write_text(json.dumps(normalized, indent=2, default=str), encoding="utf-8")
    log.info(f"JSON saved: {json_path}")
    source_counts = Counter(r["source_dataset"] for r in records)
    split_counts  = Counter(r.get("split", "?") for r in records)
    bw_count      = sum(1 for r in records if r.get("color_mode") in ("L", "1", "P"))
    hist_count    = sum(1 for r in records if r.get("historical_flag"))
    color_hist    = sum(1 for r in records if r.get("historical_color_flag"))
    extreme_asp   = sum(1 for r in records if r.get("extreme_aspect"))
    summary = {
        "generated_at": datetime.now().isoformat(), "mode": cfg["mode"],
        "total_images": len(records), "per_source_counts": dict(source_counts),
        "per_split_counts": dict(split_counts), "historical_images": hist_count,
        "historical_color": color_hist, "bw_images": bw_count,
        "extreme_aspect_flagged": extreme_asp, "split_ratios": cfg["split"],
    }
    (mdir / f"{cfg['summary_name']}.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    lines = [
        "=" * 60, "DATASET SUMMARY REPORT",
        f"Generated : {summary['generated_at']}",
        f"Mode      : {cfg['mode'].upper()}  |  Total: {len(records)} images",
        "=" * 60, "\nPer-Source:",
        *[f"  {s:<25} {c:>6}" for s, c in sorted(source_counts.items())],
        "\nPer-Split:",
        *[f"  {s:<10} {c:>6}" for s, c in sorted(split_counts.items())],
        f"\nHistorical        : {hist_count}",
        f"Historical+color  : {color_hist}",
        f"B&W / monochrome  : {bw_count}",
        f"Extreme aspect    : {extreme_asp}",
        "=" * 60,
    ]
    report = "\n".join(lines)
    (mdir / "per_source_report.txt").write_text(report, encoding="utf-8")
    print(report)


print("Processing functions defined OK")


Processing functions defined OK


In [ ]:
# ── CELL 8 ── Run the pipeline
def run_pipeline(cfg: dict = CONFIG) -> List[dict]:
    mode   = cfg["mode"]
    budget = cfg["budgets"][mode]
    log.info(f"Pipeline start — mode={mode}, budget={budget}")

    quotas = {src: max(1, int(budget * frac)) for src, frac in cfg["source_fractions"].items()}
    log.info(f"Quotas: {quotas}")

    loader_map = {
        "imagenet":         load_imagenet,
        "places365":        load_places365,
        "coco":             load_coco,
        "flickr_commons":   load_flickr_commons,
        "loc":              load_loc,
        "british_library":  load_british_library,
        #"biglam_europeana": load_biglam,
    }

    all_records: List[dict] = []
    source_counts_loaded: Dict[str, int] = {}

    for source, loader in loader_map.items():
        quota = quotas.get(source, 0)
        if quota == 0:
            continue
        log.info(f"--- Loading {source} (quota={quota}) ---")
        try:
            recs = loader(cfg, quota)
            all_records.extend(recs)
            source_counts_loaded[source] = len(recs)
            log.info(f"    {source}: {len(recs)} records loaded")
        except Exception as e:
            log.error(f"Loader FAILED [{source}]: {e}", exc_info=True)
            source_counts_loaded[source] = 0

    log.info(f"Loaded before validation: {len(all_records)}")
    log.info(f"Per-source loaded: {source_counts_loaded}")

    if len(all_records) == 0:
        log.error("NO RECORDS LOADED — check loader warnings above")
        return []

    log.info("--- Validating & deduplicating ---")
    all_records = process_records(all_records, cfg, PATHS)

    log.info("--- Edge-case boosting ---")
    all_records = boost_edge_cases(all_records, cfg)

    log.info("--- Assigning splits ---")
    all_records = assign_splits(all_records, cfg)

    log.info("--- Saving manifests ---")
    save_manifests(all_records, cfg, PATHS)

    # Final file count verification — confirms files are actually on Drive
    total_files = sum(
        len(list(p.glob("*")))
        for key, p in PATHS.items() if key.startswith("img_")
    )
    log.info(f"FILES ON DRIVE: {total_files} image files confirmed in {PATHS['images']}")
    log.info(f"Pipeline complete — {len(all_records)} records")
    log.info(f"Output: {PATHS['root']}")
    return all_records


# Run it
final_records = run_pipeline(CONFIG)
print(f"\nDone. {len(final_records)} images in manifest.")
print(f"Images folder: {PATHS['images']}")
print(f"Manifest:      {PATHS['manifests']}")

# Count actual files written to Drive
for key, p in PATHS.items():
    if key.startswith("img_"):
        n = len(list(p.glob("*")))
        if n > 0:
            print(f"  {key}: {n} files")

[12:22:36] INFO — Pipeline start — mode=medium, budget=6000


INFO:pipeline:Pipeline start — mode=medium, budget=6000


[12:22:36] INFO — Quotas: {'imagenet': 1020, 'places365': 1080, 'coco': 1800, 'british_library': 900, 'loc': 900, 'flickr_commons': 300}


INFO:pipeline:Quotas: {'imagenet': 1020, 'places365': 1080, 'coco': 1800, 'british_library': 900, 'loc': 900, 'flickr_commons': 300}


[12:22:36] INFO — --- Loading imagenet (quota=1020) ---


...

[12:36:39] INFO — Pipeline complete — 3957 records


INFO:pipeline:Pipeline complete — 3957 records


[12:36:39] INFO — Output: /content/drive/MyDrive/merged_dataset


INFO:pipeline:Output: /content/drive/MyDrive/merged_dataset



Done. 3957 images in manifest.
Images folder: /content/drive/MyDrive/merged_dataset/images
Manifest:      /content/drive/MyDrive/merged_dataset/manifests
  img_imagenet: 1440 files
  img_places365: 1080 files
  img_coco: 1800 files
  img_loc: 110 files


In [ ]:
# ── CELL 9 ── CelebA Portrait Dataset (standalone)
# Loads CelebA faces into its own separate folder and manifest.
# Independent from the main pipeline.
# Run AFTER Cell 4 (so log and PATHS are already set).

CELEBA_CONFIG = {
    "hf_dataset":   "nielsr/CelebA-faces",
    "quota":        1000,          # how many portraits to download
    "output_dir":   "/content/drive/MyDrive/merged_dataset/images/celeba/",
    "manifest_dir": "/content/drive/MyDrive/merged_dataset/manifests/",
    "min_width":    128,
    "min_height":   128,
}


def load_celeba(cfg: dict) -> None:
    import os, csv, json, hashlib, random
    from pathlib import Path
    from datetime import datetime
    from datasets import load_dataset
    from PIL import Image
    from tqdm.auto import tqdm

    img_dir  = Path(cfg["output_dir"])
    mdir     = Path(cfg["manifest_dir"])
    img_dir.mkdir(parents=True, exist_ok=True)
    mdir.mkdir(parents=True, exist_ok=True)

    # Drive write test
    test = img_dir / "_write_test.txt"
    test.write_text("ok"); assert test.exists(); test.unlink()
    log.info("CelebA: Drive write test PASSED")

    # Load dataset
    log.info(f"CelebA: loading {cfg['hf_dataset']}")
    try:
        ds = load_dataset(cfg["hf_dataset"], split="train")
    except Exception as e:
        log.warning(f"CelebA load failed: {e}")
        return

    log.info(f"CelebA: {len(ds)} total rows. Columns: {list(ds[0].keys())}")

    # Probe image column
    sample = ds[0]
    img_col = next((c for c in ["image", "img", "pixel_values"]
                    if c in sample and sample[c] is not None), None)
    if img_col is None:
        log.warning("CelebA: no image column found — aborting")
        return

    # Sample
    indices = list(range(len(ds)))
    random.seed(42); random.shuffle(indices)
    chosen = indices[:cfg["quota"]]

    records = []
    saved   = 0

    for i in tqdm(chosen, desc="CelebA"):
        row = ds[i]
        val = row.get(img_col)
        if val is None:
            continue

        # Convert to PIL
        if isinstance(val, Image.Image):
            img = val
        else:
            try:
                import numpy as np
                img = Image.fromarray(val)
            except Exception:
                continue

        # Validate size
        w, h = img.size
        if w < cfg["min_width"] or h < cfg["min_height"]:
            continue

        # Save
        dst = str(img_dir / f"celeba_{i}.jpg")
        try:
            img.convert("RGB").save(dst, quality=95)
            saved += 1
        except Exception as e:
            log.debug(f"CelebA save failed idx {i}: {e}")
            continue

        records.append({
            "image_id":              f"celeba_{hashlib.md5(dst.encode()).hexdigest()[:10]}",
            "source_dataset":        "celeba",
            "source_subset":         "CelebA-faces",
            "original_path_or_url":  f"hf://{cfg['hf_dataset']}#{i}",
            "local_path":            dst,
            "split":                 "train",
            "width":                 w,
            "height":                h,
            "aspect_ratio":          round(w / h, 3),
            "color_mode":            img.mode,
            "file_size":             os.path.getsize(dst),
            "label":                 "portrait",
            "scene_type":            "portrait",
            "historical_flag":       False,
            "historical_color_flag": False,
            "damaged_flag":          False,
            "extreme_aspect":        False,
            "license_info":          "CelebA — non-commercial research only",
            "institution":           "CUHK",
            "decade":                None,
            "notes":                 "Celebrity face portrait",
        })

    log.info(f"CelebA: {saved} files saved to {img_dir}")

    if not records:
        log.warning("CelebA: no records to save — check warnings above")
        return

    # Save manifest
    fields = list(records[0].keys())

    csv_path = mdir / "celeba_manifest.csv"
    with open(str(csv_path), "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)
    log.info(f"CelebA CSV: {csv_path}")

    json_path = mdir / "celeba_manifest.json"
    json_path.write_text(json.dumps(records, indent=2, default=str), encoding="utf-8")
    log.info(f"CelebA JSON: {json_path}")

    # Summary
    print(f"""
============================================================
CELEBA SUMMARY
============================================================
Dataset : {cfg['hf_dataset']}
Quota   : {cfg['quota']}
Saved   : {saved} images
Folder  : {img_dir}
Manifest: {csv_path}
============================================================
""")


# Run it
load_celeba(CELEBA_CONFIG)

In [ ]:
# ── CELL 12 ── Biglam / Europeana historical color images (standalone)

BIGLAM_CONFIG = {
    "hf_dataset": "huggan/wikiart",
    "quota":        1000,   # full dataset is ~1375 images
    "output_dir":   "/content/drive/MyDrive/merged_dataset/images/biglam_europeana/",
    "manifest_dir": "/content/drive/MyDrive/merged_dataset/manifests/",
    "min_width":    128,
    "min_height":   128,
    "skip_existing": True,
}


def load_biglam_standalone(cfg: dict) -> None:
    import os, csv, json, hashlib, random
    from pathlib import Path
    from datetime import datetime
    from datasets import load_dataset
    from PIL import Image
    from tqdm.auto import tqdm

    img_dir = Path(cfg["output_dir"])
    mdir    = Path(cfg["manifest_dir"])
    img_dir.mkdir(parents=True, exist_ok=True)
    mdir.mkdir(parents=True, exist_ok=True)

    # Drive write test
    test = img_dir / "_write_test.txt"
    test.write_text("ok"); assert test.exists(); test.unlink()
    log.info("Biglam: Drive write test PASSED")

    # Load dataset
    log.info(f"Biglam: loading {cfg['hf_dataset']}")
    try:
        ds = load_dataset(cfg["hf_dataset"], split="train")
    except Exception as e:
        log.warning(f"Biglam load failed: {e}")
        return

    log.info(f"Biglam: {len(ds)} total rows. Columns: {list(ds[0].keys())}")

    # Probe columns
    sample    = ds[0]
    img_col   = next((c for c in ["image", "img", "pixel_values"]
                      if c in sample and sample[c] is not None), None)
    decade_col = next((c for c in ["decade", "year", "date", "period"]
                       if c in sample and sample[c] is not None), None)

    if img_col is None:
        log.warning("Biglam: no image column found — aborting")
        return

    log.info(f"Biglam: image column='{img_col}', decade column='{decade_col}'")

    # Sample
    indices = list(range(len(ds)))
    random.seed(42); random.shuffle(indices)
    chosen = indices[:cfg["quota"]]

    records = []
    saved   = 0
    skipped = 0
    failed  = 0

    for i in tqdm(chosen, desc="Biglam"):
        row = ds[i]
        val = row.get(img_col)
        if val is None:
            continue

        # Convert to PIL
        if isinstance(val, Image.Image):
            img = val
        else:
            try:
                import numpy as np
                img = Image.fromarray(val)
            except Exception:
                continue

        # Validate size
        w, h = img.size
        if w < cfg["min_width"] or h < cfg["min_height"]:
            continue

        # Skip if already saved
        dst = str(img_dir / f"biglam_{i}.jpg")
        if cfg["skip_existing"] and Path(dst).exists():
            skipped += 1
            continue

        # Save
        try:
            img.convert("RGB").save(dst, quality=95)
            saved += 1
        except Exception as e:
            log.debug(f"Biglam save failed idx {i}: {e}")
            failed += 1
            continue

        decade = str(row.get(decade_col, "")) if decade_col else None

        records.append({
            "image_id":              f"biglam_{hashlib.md5(dst.encode()).hexdigest()[:10]}",
            "source_dataset":        "biglam_europeana",
            "source_subset":         "dating-historical-color-images",
            "original_path_or_url":  f"hf://{cfg['hf_dataset']}#{i}",
            "local_path":            dst,
            "split":                 "train",
            "width":                 w,
            "height":                h,
            "aspect_ratio":          round(w / h, 3),
            "color_mode":            "RGB",
            "file_size":             os.path.getsize(dst),
            "label":                 decade or None,
            "scene_type":            "historical",
            "historical_flag":       True,
            "historical_color_flag": True,
            "damaged_flag":          None,
            "extreme_aspect":        False,
            "license_info":          "Europeana / various open licenses",
            "institution":           "Europeana / Flickr Commons",
            "decade":                decade or None,
            "notes":                 "Pre-1980 curated color image",
        })

    log.info(f"Biglam: {saved} saved, {skipped} skipped, {failed} failed")

    if not records:
        log.warning("Biglam: no records saved")
        return

    # Save manifest
    fields    = list(records[0].keys())
    csv_path  = mdir / "biglam_manifest.csv"
    json_path = mdir / "biglam_manifest.json"

    with open(str(csv_path), "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)

    Path(json_path).write_text(
        json.dumps(records, indent=2, default=str), encoding="utf-8"
    )

    print(f"""
============================================================
BIGLAM SUMMARY
============================================================
Dataset   : {cfg['hf_dataset']}
Quota     : {cfg['quota']}
Saved     : {saved} images
Skipped   : {skipped} (already existed)
Failed    : {failed}
Folder    : {img_dir}
Manifest  : {csv_path}
============================================================
""")


# Run it
load_biglam_standalone(BIGLAM_CONFIG)

[14:14:59] INFO — Biglam: Drive write test PASSED


INFO:pipeline:Biglam: Drive write test PASSED


[14:14:59] INFO — Biglam: loading huggan/wikiart


INFO:pipeline:Biglam: loading huggan/wikiart


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

...

INFO:pipeline:Biglam: image column='image', decade column='None'


Biglam:   0%|          | 0/1000 [00:00<?, ?it/s]

[14:48:56] INFO — Biglam: 1000 saved, 0 skipped, 0 failed


INFO:pipeline:Biglam: 1000 saved, 0 skipped, 0 failed



BIGLAM SUMMARY
Dataset   : huggan/wikiart
Quota     : 1000
Saved     : 1000 images
Skipped   : 0 (already existed)
Failed    : 0
Folder    : /content/drive/MyDrive/merged_dataset/images/biglam_europeana
Manifest  : /content/drive/MyDrive/merged_dataset/manifests/biglam_manifest.csv



In [ ]:
# ── CELL 10 ── Resize all images to 256×256 and save to merged_dataset_resized

RESIZE_CONFIG = {
    "input_root":  "/content/drive/MyDrive/merged_dataset/images/",
    "output_root": "/content/drive/MyDrive/merged_dataset_resized/",
    "target_size": (256, 256),
    "quality":     95,
    "skip_existing": True,   # set False to re-process everything from scratch
}


def resize_all_images(cfg: dict) -> None:
    import os
    from pathlib import Path
    from PIL import Image
    from tqdm.auto import tqdm

    input_root  = Path(cfg["input_root"])
    output_root = Path(cfg["output_root"])
    size        = cfg["target_size"]

    # Create output folder and Drive write test
    output_root.mkdir(parents=True, exist_ok=True)
    test = output_root / "_write_test.txt"
    test.write_text("ok"); assert test.exists(); test.unlink()
    log.info(f"Resize: output folder ready — {output_root}")

    # Collect all images recursively from all subfolders
    extensions = {".jpg", ".jpeg", ".png"}
    all_images = [
        p for p in input_root.rglob("*")
        if p.suffix.lower() in extensions and p.is_file()
    ]
    log.info(f"Resize: found {len(all_images)} images to process")

    if len(all_images) == 0:
        log.warning("Resize: no images found in input folder — check path")
        return

    saved    = 0
    skipped  = 0
    failed   = 0

    for src_path in tqdm(all_images, desc="Resizing"):
        # Build a flat unique filename: sourcefolder_originalname.jpg
        # e.g. imagenet_imagenet_123.jpg, celeba_celeba_456.jpg
        source_folder = src_path.parent.name
        flat_name     = f"{source_folder}_{src_path.stem}.jpg"
        dst_path      = output_root / flat_name

        # Skip if already done
        if cfg["skip_existing"] and dst_path.exists():
            skipped += 1
            continue

        try:
            with Image.open(src_path) as img:
                # Convert to RGB (handles grayscale, RGBA, palette modes)
                img_rgb = img.convert("RGB")
                # Resize with high quality resampling
                img_resized = img_rgb.resize(size, Image.LANCZOS)
                img_resized.save(str(dst_path), "JPEG", quality=cfg["quality"])
                saved += 1
        except Exception as e:
            log.debug(f"Resize failed {src_path}: {e}")
            failed += 1
            continue

    print(f"""
============================================================
RESIZE COMPLETE
============================================================
Input folder  : {input_root}
Output folder : {output_root}
Target size   : {size[0]}×{size[1]}
------------------------------------------------------------
Processed     : {saved}
Skipped       : {skipped}  (already existed)
Failed        : {failed}   (corrupted/unreadable)
Total         : {len(all_images)}
============================================================
""")
    log.info(f"Resize complete — {saved} saved, {skipped} skipped, {failed} failed")


# Run it
resize_all_images(RESIZE_CONFIG)

[14:50:45] INFO — Resize: output folder ready — /content/drive/MyDrive/merged_dataset_resized


INFO:pipeline:Resize: output folder ready — /content/drive/MyDrive/merged_dataset_resized


[14:50:47] INFO — Resize: found 6430 images to process


INFO:pipeline:Resize: found 6430 images to process


Resizing:   0%|          | 0/6430 [00:00<?, ?it/s]


RESIZE COMPLETE
Input folder  : /content/drive/MyDrive/merged_dataset/images
Output folder : /content/drive/MyDrive/merged_dataset_resized
Target size   : 256×256
------------------------------------------------------------
Processed     : 1000
Skipped       : 5430  (already existed)
Failed        : 0   (corrupted/unreadable)
Total         : 6430

[14:52:27] INFO — Resize complete — 1000 saved, 5430 skipped, 0 failed


INFO:pipeline:Resize complete — 1000 saved, 5430 skipped, 0 failed


In [ ]:
!fusermount -u /content/drive

fusermount: failed to unmount /content/drive: Invalid argument


In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ── CELL 11 ── Convert resized images to grayscale and save to merged_dataset_bw

BW_CONFIG = {
    "input_root":    "/content/drive/MyDrive/merged_dataset_resized/",
    "output_root":   "/content/drive/MyDrive/merged_dataset_bw/",
    "skip_existing": True,   # set False to re-process everything from scratch
}

def convert_to_grayscale(cfg: dict) -> None:
    from pathlib import Path
    from PIL import Image
    from tqdm.auto import tqdm

    input_root  = Path(cfg["input_root"])
    output_root = Path(cfg["output_root"])

    # Create output folder and Drive write test
    output_root.mkdir(parents=True, exist_ok=True)
    test = output_root / "_write_test.txt"
    test.write_text("ok"); assert test.exists(); test.unlink()
    log.info(f"BW: output folder ready — {output_root}")

    # Collect all images from the flat resized folder
    extensions = {".jpg", ".jpeg", ".png"}
    all_images = [
        p for p in input_root.iterdir()
        if p.suffix.lower() in extensions and p.is_file()
    ]
    log.info(f"BW: found {len(all_images)} images to convert")

    if len(all_images) == 0:
        log.warning("BW: no images found — check that Cell 10 ran successfully")
        return

    saved   = 0
    skipped = 0
    failed  = 0

    for src_path in tqdm(all_images, desc="Grayscale"):
        dst_path = output_root / src_path.name  # same filename, different folder

        if cfg["skip_existing"] and dst_path.exists():
            skipped += 1
            continue

        try:
            with Image.open(src_path) as img:
                # BT.709 luminance formula — perceptually accurate grayscale
                # R*0.2126 + G*0.7152 + B*0.0722
                img_gray = img.convert("RGB").convert("L")
                img_gray.save(str(dst_path), "JPEG", quality=95)
                saved += 1
        except Exception as e:
            log.debug(f"BW conversion failed {src_path}: {e}")
            failed += 1
            continue

    print(f"""
============================================================
GRAYSCALE CONVERSION COMPLETE
============================================================
Input folder  : {input_root}
Output folder : {output_root}
------------------------------------------------------------
Converted     : {saved}
Skipped       : {skipped}  (already existed)
Failed        : {failed}   (corrupted/unreadable)
Total         : {len(all_images)}
============================================================
""")
    log.info(f"BW complete — {saved} converted, {skipped} skipped, {failed} failed")


# Run it
convert_to_grayscale(BW_CONFIG)


[14:52:34] INFO — BW: output folder ready — /content/drive/MyDrive/merged_dataset_bw


INFO:pipeline:BW: output folder ready — /content/drive/MyDrive/merged_dataset_bw


[14:52:36] INFO — BW: found 6430 images to convert


INFO:pipeline:BW: found 6430 images to convert


Grayscale:   0%|          | 0/6430 [00:00<?, ?it/s]


GRAYSCALE CONVERSION COMPLETE
Input folder  : /content/drive/MyDrive/merged_dataset_resized
Output folder : /content/drive/MyDrive/merged_dataset_bw
------------------------------------------------------------
Converted     : 1000
Skipped       : 5430  (already existed)
Failed        : 0   (corrupted/unreadable)
Total         : 6430

[14:54:26] INFO — BW complete — 1000 converted, 5430 skipped, 0 failed


INFO:pipeline:BW complete — 1000 converted, 5430 skipped, 0 failed
